# Download and normalize financial sentiment datasets

This notebook pulls the four datasets referenced for the project and writes canonical processed files under `data/processed/`.

Sources covered:
- Kaggle: `sbhatti/financial-sentiment-analysis`
- Hugging Face: `lwrf42/financial-sentiment-dataset`
- Hugging Face: `Kenpache/multilingual-financial-sentiment` (English subset)
- Hugging Face: `maguid28/combined_financial_phrasebank_twitter_news_sentiment`

Notes:
- `Kenpache/multilingual-financial-sentiment` is filtered to the English subset and uses direct `negative/neutral/positive` sentiment labels.
- The Kaggle source is public, but Kaggle may still require local credentials through `~/.kaggle/kaggle.json` or `KAGGLE_USERNAME` + `KAGGLE_KEY`.

In [1]:
from pathlib import Path
import json
import re
import unicodedata
import warnings

import kagglehub
import pandas as pd
from html import unescape
from datasets import load_dataset
from IPython.display import display

warnings.filterwarnings("ignore")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BASE_COLUMNS = [
    "dataset_id",
    "dataset_label",
    "source_platform",
    "split",
    "text",
    "label_normalized",
]

In [2]:
POSITIVE_LABELS = {
    "positive",
    "mildly positive",
    "moderately positive",
    "strong positive",
}
NEGATIVE_LABELS = {
    "negative",
    "mildly negative",
    "moderately negative",
    "strong negative",
}
NEUTRAL_LABELS = {"neutral"}


def clean_text(value):
    text = unescape(str(value))
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def normalize_label(value):
    if pd.isna(value):
        return None
    normalized = str(value).strip().lower()
    if normalized in POSITIVE_LABELS:
        return "positive"
    if normalized in NEGATIVE_LABELS:
        return "negative"
    if normalized in NEUTRAL_LABELS:
        return "neutral"
    return normalized


def curate_columns(df):
    return df[BASE_COLUMNS].copy()


def save_canonical(df, dataset_key):
    csv_path = PROCESSED_DIR / f"{dataset_key}_canonical.csv"
    parquet_path = PROCESSED_DIR / f"{dataset_key}_canonical.parquet"
    exported = curate_columns(df.copy())
    exported.to_csv(csv_path, index=False)
    exported.to_parquet(parquet_path, index=False)
    return exported, csv_path, parquet_path


def save_raw_snapshot(df, dataset_key, split_name):
    raw_split_dir = RAW_DIR / dataset_key / split_name
    raw_split_dir.mkdir(parents=True, exist_ok=True)
    raw_snapshot_path = raw_split_dir / "raw_snapshot.csv"
    df.to_csv(raw_snapshot_path, index=False)
    return raw_snapshot_path


def build_split_payload(df, dataset_key, split_name):
    raw_snapshot_path = save_raw_snapshot(df, dataset_key, split_name)
    return {
        "dataframe": df,
        "raw_snapshot": raw_snapshot_path,
    }


def compute_split_counts(total_rows):
    test_rows = round(total_rows * 0.2)
    validation_rows = round(total_rows * 0.2)
    train_rows = total_rows - test_rows - validation_rows

    if train_rows < 0:
        validation_rows = max(validation_rows + train_rows, 0)
        train_rows = total_rows - test_rows - validation_rows
    if train_rows < 0:
        test_rows = max(test_rows + train_rows, 0)
        train_rows = total_rows - test_rows - validation_rows

    return {
        "train": train_rows,
        "validation": validation_rows,
        "test": test_rows,
    }


def derive_curated_splits(source_payload):
    preferred_order = ["train", "validation", "test"]
    ordered_split_names = [split_name for split_name in preferred_order if split_name in source_payload["splits"]]
    ordered_split_names.extend(
        split_name
        for split_name in source_payload["splits"].keys()
        if split_name not in ordered_split_names
    )
    frames = [source_payload["splits"][split_name]["dataframe"] for split_name in ordered_split_names]
    raw_df = pd.concat(frames, ignore_index=True, sort=False)
    raw_snapshot = " | ".join(str(source_payload["splits"][split_name]["raw_snapshot"]) for split_name in ordered_split_names)
    derived_from = ",".join(ordered_split_names)
    counts = compute_split_counts(len(raw_df))

    train_end = counts["train"]
    validation_end = train_end + counts["validation"]

    curated_splits = {
        "train": {
            "dataframe": raw_df.iloc[:train_end].reset_index(drop=True),
            "raw_snapshot": raw_snapshot,
            "derived_from": derived_from,
        },
        "validation": {
            "dataframe": raw_df.iloc[train_end:validation_end].reset_index(drop=True),
            "raw_snapshot": raw_snapshot,
            "derived_from": derived_from,
        },
        "test": {
            "dataframe": raw_df.iloc[validation_end:].reset_index(drop=True),
            "raw_snapshot": raw_snapshot,
            "derived_from": derived_from,
        },
    }
    return curated_splits


def standardize_lwrf(df, split_name):
    standardized = df.copy()
    standardized["dataset_id"] = "lwrf42/financial-sentiment-dataset"
    standardized["dataset_label"] = "lwrf42_financial_sentiment_dataset"
    standardized["source_platform"] = "huggingface"
    standardized["split"] = split_name
    standardized["text"] = standardized["input"].map(clean_text)
    standardized["label_raw"] = standardized["output"].astype(str).str.strip().str.lower()
    standardized["label_normalized"] = standardized["label_raw"].map(normalize_label)
    return standardized


def standardize_kenpache_english(df, split_name):
    standardized = df[df["language"] == "en"].copy()
    standardized["dataset_id"] = "Kenpache/multilingual-financial-sentiment"
    standardized["dataset_label"] = "kenpache_multilingual_financial_sentiment_en"
    standardized["source_platform"] = "huggingface"
    standardized["split"] = split_name
    standardized["text"] = standardized["sentence"].map(clean_text)
    standardized["label_raw"] = standardized["label"].astype(str).str.strip().str.lower()
    standardized["label_normalized"] = standardized["label_raw"].map(normalize_label)
    return standardized


def standardize_maguid(df, split_name):
    standardized = df.copy()
    standardized["dataset_id"] = "maguid28/combined_financial_phrasebank_twitter_news_sentiment"
    standardized["dataset_label"] = "maguid28_combined_financial_phrasebank_twitter_news_sentiment"
    standardized["source_platform"] = "huggingface"
    standardized["split"] = split_name
    standardized["text"] = standardized["text"].map(clean_text)
    standardized["label_raw"] = standardized["polarity"].astype(str).str.strip().str.lower()
    standardized["label_normalized"] = standardized["label_raw"].map(normalize_label)
    return standardized


def standardize_kaggle(df, split_name):
    standardized = df.copy()
    standardized["dataset_id"] = "sbhatti/financial-sentiment-analysis"
    standardized["dataset_label"] = "sbhatti_financial_sentiment_analysis"
    standardized["source_platform"] = "kaggle"
    standardized["split"] = split_name
    standardized["text"] = standardized["Sentence"].map(clean_text)
    standardized["label_raw"] = standardized["Sentiment"].astype(str).str.strip().str.lower()
    standardized["label_normalized"] = standardized["label_raw"].map(normalize_label)
    return standardized


def summarize_frame(df):
    return {
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "unique_raw_labels": int(df["label_raw"].nunique(dropna=True)),
        "unique_normalized_labels": int(df["label_normalized"].nunique(dropna=True)),
    }


def load_huggingface_source(config):
    dataset = load_dataset(config["repo_id"], verification_mode="no_checks")
    splits = {}
    for split_name, split_dataset in dataset.items():
        raw_df = split_dataset.to_pandas()
        splits[split_name] = build_split_payload(raw_df, config["key"], split_name)

    return {"splits": splits}


def load_kaggle_source(config):
    kaggle_download_root = Path(kagglehub.dataset_download(config["repo_id"]))
    csv_candidates = sorted(kaggle_download_root.rglob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError("No CSV files were found inside the Kaggle download.")

    raw_frames = [pd.read_csv(csv_path) for csv_path in csv_candidates]
    raw_df = pd.concat(raw_frames, ignore_index=True, sort=False)
    split_name = config.get("default_split", "train")
    return {
        "splits": {
            split_name: build_split_payload(raw_df, config["key"], split_name)
        },
    }


def process_source(config, manifest, combined_frames):
    try:
        source_payload = config["loader"](config)
        curated_splits = derive_curated_splits(source_payload)
        dataset_frames = []

        for split_name, split_payload in curated_splits.items():
            standardized = config["standardizer"](
                split_payload["dataframe"],
                split_name,
            )
            dataset_frames.append(standardized)
            split_summary = summarize_frame(standardized)
            manifest.append(
                {
                    "dataset_key": config["key"],
                    "dataset_id": config["repo_id"],
                    "source_platform": config["source_platform"],
                    "status": "downloaded",
                    "split": split_name,
                    "rows": split_summary["rows"],
                    "columns": split_summary["columns"],
                    "unique_raw_labels": split_summary["unique_raw_labels"],
                    "unique_normalized_labels": split_summary["unique_normalized_labels"],
                    "raw_snapshot": str(split_payload["raw_snapshot"]),
                    "derived_from": split_payload.get("derived_from"),
                }
            )

        dataset_frame = pd.concat(dataset_frames, ignore_index=True, sort=False)
        exported, csv_path, parquet_path = save_canonical(dataset_frame, config["key"])
        combined_frames.append(exported)
        manifest.append(
            {
                "dataset_key": config["key"],
                "dataset_id": config["repo_id"],
                "source_platform": config["source_platform"],
                "status": "exported",
                "split": "all",
                "rows": int(len(exported)),
                "columns": int(len(exported.columns)),
                "csv_path": str(csv_path),
                "parquet_path": str(parquet_path),
            }
        )
    except Exception as exc:
        manifest.append(
            {
                "dataset_key": config["key"],
                "dataset_id": config["repo_id"],
                "source_platform": config["source_platform"],
                "status": "failed",
                "split": config.get("default_split", "train"),
                "error": str(exc),
            }
        )
        print(f"Source load failed for {config['repo_id']}: {exc}")


In [3]:
source_configs = [
    {
        "key": "lwrf42_financial_sentiment_dataset",
        "repo_id": "lwrf42/financial-sentiment-dataset",
        "source_platform": "huggingface",
        "loader": load_huggingface_source,
        "standardizer": standardize_lwrf,
    },
    {
        "key": "kenpache_multilingual_financial_sentiment",
        "repo_id": "Kenpache/multilingual-financial-sentiment",
        "source_platform": "huggingface",
        "loader": load_huggingface_source,
        "standardizer": standardize_kenpache_english,
    },
    {
        "key": "maguid28_combined_financial_phrasebank_twitter_news_sentiment",
        "repo_id": "maguid28/combined_financial_phrasebank_twitter_news_sentiment",
        "source_platform": "huggingface",
        "loader": load_huggingface_source,
        "standardizer": standardize_maguid,
    },
    {
        "key": "sbhatti_financial_sentiment_analysis",
        "repo_id": "sbhatti/financial-sentiment-analysis",
        "source_platform": "kaggle",
        "loader": load_kaggle_source,
        "standardizer": standardize_kaggle,
        "default_split": "train",
    },
]

manifest = []
combined_frames = []

for config in source_configs:
    process_source(config, manifest, combined_frames)

display(pd.DataFrame(manifest))


,dataset_key,dataset_id,source_platform,status,split,rows,columns,unique_raw_labels,unique_normalized_labels,raw_snapshot,derived_from,csv_path,parquet_path
0,lwrf42_financial_sentiment_dataset,lwrf42/financial-sentiment-dataset,huggingface,downloaded,train,57132,11,9.0,3.0,/home/automac/Documents/Estudios/Semestre3/Pro...,"train,validation",NaN,NaN
1,lwrf42_financial_sentiment_dataset,lwrf42/financial-sentiment-dataset,huggingface,downloaded,validation,19044,11,9.0,3.0,/home/automac/Documents/Estudios/Semestre3/Pro...,"train,validation",NaN,NaN
2,lwrf42_financial_sentiment_dataset,lwrf42/financial-sentiment-dataset,huggingface,downloaded,test,19044,11,9.0,3.0,/home/automac/Documents/Estudios/Semestre3/Pro...,"train,validation",NaN,NaN
3,lwrf42_financial_sentiment_dataset,lwrf42/financial-sentiment-dataset,huggingface,exported,all,95220,6,NaN,NaN,NaN,NaN,/home/automac/Documents/Estudios/Semestre3/Pro...,/home/automac/Documents/Estudios/Semestre3/Pro...
4,kenpache_multilingual_financial_sentiment,Kenpache/multilingual-financial-sentiment,huggingface,downloaded,train,6887,11,3.0,3.0,/home/automac/Documents/Estudios/Semestre3/Pro...,train,NaN,NaN
5,kenpache_multilingual_financial_sentiment,Kenpache/multilingual-financial-sentiment,huggingface,downloaded,validation,0,11,0.0,0.0,/home/automac/Documents/Estudios/Semestre3/Pro...,train,NaN,NaN
6,kenpache_multilingual_financial_sentiment,Kenpache/multilingual-financial-sentiment,huggingface,downloaded,test,0,11,0.0,0.0,/home/automac/Documents/Estudios/Semestre3/Pro...,train,NaN,NaN
7,kenpache_multilingual_financial_sentiment,Kenpache/multilingual-financial-sentiment,huggingface,exported,all,6887,6,NaN,NaN,NaN,NaN,/home/automac/Documents/Estudios/Semestre3/Pro...,/home/automac/Documents/Estudios/Semestre3/Pro...
8,maguid28_combined_financial_phrasebank_twitter...,maguid28/combined_financial_phrasebank_twitter...,huggingface,downloaded,train,10067,8,3.0,3.0,/home/automac/Documents/Estudios/Semestre3/Pro...,train,NaN,NaN
9,maguid28_combined_financial_phrasebank_twitter...,maguid28/combined_financial_phrasebank_twitter...,huggingface,downloaded,validation,3355,8,3.0,3.0,/home/automac/Documents/Estudios/Semestre3/Pro...,train,NaN,NaN


In [4]:
if not combined_frames:
    raise RuntimeError("No datasets were exported. Check the dataset source configuration.")

combined = pd.concat(combined_frames, ignore_index=True, sort=False)
combined = combined[combined["text"].notna()].copy()
combined["text"] = combined["text"].astype(str).str.strip()
combined = combined[combined["text"] != ""].copy()
combined = combined[combined["label_normalized"].isin(["negative", "neutral", "positive"])].copy()
combined_csv = PROCESSED_DIR / "financial_sentiment_combined.csv"
combined_parquet = PROCESSED_DIR / "financial_sentiment_combined.parquet"
combined.to_csv(combined_csv, index=False)
combined.to_parquet(combined_parquet, index=False)

manifest_path = PROCESSED_DIR / "download_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(combined.head(10))
print(f"Combined CSV written to: {combined_csv}")
print(f"Combined Parquet written to: {combined_parquet}")


,dataset_id,dataset_label,source_platform,split,text,label_normalized
0,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,general announcement monthly valuation of asse...,neutral
1,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,cowen survey forecasts stores to reopen in jun...,positive
2,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,aap weak no volume,negative
3,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,rt robbielolz nflx a close above here is looki...,positive
4,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,apple s aapl third quarter fiscal earnings ben...,positive
5,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,new product launches in finland will more than...,positive
6,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,cpst yndx among premarket gainers,positive
7,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,aapl bounces off support it seems,positive
8,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,the airline s share price closed down slightly...,negative
9,lwrf42/financial-sentiment-dataset,lwrf42_financial_sentiment_dataset,huggingface,train,ung ugaz dgaz freeport lng launches commercial...,neutral


Combined CSV written to: /home/automac/Documents/Estudios/Semestre3/ProyectoDeGrado/data/processed/financial_sentiment_combined.csv
Combined Parquet written to: /home/automac/Documents/Estudios/Semestre3/ProyectoDeGrado/data/processed/financial_sentiment_combined.parquet
